In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import numpy as np
import statsmodels.api as sm
from statistics import t_test, t_test_binary_col, anova

In [5]:
font_path = "C:/Windows/Fonts/msjh.ttc"
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei']  # 微軟正黑體
plt.rcParams['axes.unicode_minus'] = False  # 正負號也要支援中文

# EDA

In [6]:
data = pd.read_csv('LDA topics.csv')
 # 薪資正規化
data['max_salary_K'] = data['max_salary'] / 1000
data['min_salary_K'] = data['min_salary'] / 1000

In [7]:
salary_data = data[(data['max_salary'] > 0)&(data['max_salary'] < 500000)&(data['salaryType'] != 'H')&(data['salaryType'] != 'D')].copy()
analyst_data = salary_data[(salary_data['topic_job_category']=='BA')|((salary_data['topic_job_category']=='DA')|((salary_data['topic_job_category']=='DS')))].copy()

## 1. 產業 v.s. 薪資

In [8]:
industries = list(salary_data['coIndustryDesc'].unique())
t_test(salary_data, 'coIndustryDesc', industries, 'max_salary_K', 0.1)

,coIndustryDesc,mean max_salary_K_with,mean max_salary_K_not_with,mean_diff,p_value
6,工商顧問服務業,55.547619,81.121349,-25.573730,3.084588e-08
3,電腦軟體服務業,99.520868,74.757592,24.763276,1.099829e-04
4,醫院,59.006923,80.378608,-21.371685,3.273061e-04


In [9]:
salary_data['industry_group_no'] = salary_data['coIndustry'].astype(str).str[:7]
industries_no = list(salary_data['industry_group_no'].unique())
test_df = t_test(salary_data, 'industry_group_no', industries_no, 'max_salary_K', 0.05)

In [10]:
industry_map = {
    '1001001': '電腦軟體相關產業',
    '1001003': '電腦設備相關產業',
    '1012001': '醫療業',
    '1008003': '工商顧問及其他專業'
}

test_df['industry_group'] = test_df['industry_group_no'].apply(lambda x: industry_map[x])
test_df

,industry_group_no,mean max_salary_K_with,mean max_salary_K_not_with,mean_diff,p_value,industry_group
2,1001003,60.545455,80.202078,-19.656623,1.370384e-08,電腦設備相關產業
0,1001001,90.047270,70.305676,19.741595,5.396879e-05,電腦軟體相關產業
4,1012001,59.077857,80.439440,-21.361583,1.294801e-04,醫療業
5,1008003,61.779677,81.318026,-19.538349,4.088871e-04,工商顧問及其他專業


In [11]:
test_df = t_test(salary_data, 'industry_group_no', industries_no, 'min_salary_K', 0.05)

In [12]:
test_df['industry_group'] = test_df['industry_group_no'].apply(lambda x: industry_map[x])
test_df

,industry_group_no,mean min_salary_K_with,mean min_salary_K_not_with,mean_diff,p_value,industry_group
2,1001003,32.818182,50.300108,-17.481927,2.179097e-24,電腦設備相關產業
0,1001001,56.162343,44.063850,12.098493,9.401251e-07,電腦軟體相關產業
4,1012001,40.980357,50.115557,-9.135200,1.004689e-04,醫療業


## 2. 學歷 v.s 薪資

In [13]:
edu = list(salary_data['optionEdu'].unique())
print(edu)
test_df = t_test(salary_data, 'optionEdu', edu, 'max_salary_K', 0.05)
test_df

['大學', '學歷不拘', '碩士', '專科', '高中', '高中以下', '博士']


,optionEdu,mean max_salary_K_with,mean max_salary_K_not_with,mean_diff,p_value
3,專科,65.339461,83.029249,-17.689788,0.000148


In [14]:
edu = list(salary_data['optionEdu'].unique())
test_df = t_test(salary_data, 'optionEdu', edu, 'min_salary_K', 0.05)
test_df

,optionEdu,mean min_salary_K_with,mean min_salary_K_not_with,mean_diff,p_value
3,專科,43.191520,51.335991,-8.144471,0.000309
0,大學,52.239699,46.521158,5.718541,0.014011


## 3. 學系要求 v.s. 薪資

In [15]:
majors = ['major_資訊工程相關', 'major_資訊管理相關',
       'major_電機電子工程相關', 'major_數學及電算機科學學科類', 'major_數理統計相關',
       'major_其他數學及電算機科學相關', 'major_統計學相關', 'major_商業及管理學科類', 'major_工程學科類',
       'major_應用數學相關', 'major_財稅金融相關', 'major_工業工程相關', 'major_一般商業學類',
       'major_會計學相關', 'major_機械工程相關']
test_df = t_test_binary_col(salary_data, majors, 'min_salary_K', 0.05)
test_df

,name,mean min_salary_K_with,mean min_salary_K_not_with,mean_diff,p_value
6,major_統計學相關,38.130208,50.921987,-12.791779,1.063955e-08
4,major_數理統計相關,39.937037,50.571853,-10.634816,9.964822e-08
1,major_資訊管理相關,40.609087,51.685008,-11.075921,6.602039e-07


In [16]:
test_df = t_test_binary_col(salary_data, majors, 'max_salary_K', 0.05)
test_df

,name,mean max_salary_K_with,mean max_salary_K_not_with,mean_diff,p_value
6,major_統計學相關,57.572917,81.804319,-24.231402,5.107862e-09
4,major_數理統計相關,60.977778,81.142554,-20.164776,1.641066e-07
1,major_資訊管理相關,65.188842,82.631191,-17.442349,2.076348e-05
2,major_電機電子工程相關,103.185417,76.527138,26.658279,2.602563e-02
5,major_其他數學及電算機科學相關,106.786576,77.752286,29.034289,3.881663e-02


In [17]:
salary_data['major_group_infomation'] =  salary_data[['major_資訊工程相關', 'major_資訊工程相關']].sum(axis=1).gt(0).astype(int)
salary_data['major_group_engineer'] = salary_data[['major_電機電子工程相關', 'major_工程學科類', 'major_機械工程相關']].sum(axis=1).gt(0).astype(int)
salary_data['major_group_math_and_stats'] =  salary_data[['major_數學及電算機科學學科類', 'major_數理統計相關', 'major_其他數學及電算機科學相關', 'major_應用數學相關', 'major_統計學相關']].sum(axis=1).gt(0).astype(int)
salary_data['major_group_commercial'] = salary_data[['major_商業及管理學科類', 'major_一般商業學類']].sum(axis=1).gt(0).astype(int)

major_group = ['major_group_infomation', 'major_group_engineer', 'major_group_math_and_stats', 'major_group_commercial']
test_df = t_test_binary_col(salary_data, major_group, 'max_salary_K', 0.1)
test_df 

,name,mean max_salary_K_with,mean max_salary_K_not_with,mean_diff,p_value
3,major_group_commercial,58.070000,80.217116,-22.147116,0.003614
1,major_group_engineer,99.988562,76.090220,23.898342,0.014460


In [18]:
salary_data['major_group_infomation'] =  salary_data[['major_資訊工程相關', 'major_資訊工程相關']].sum(axis=1).gt(0).astype(int)
salary_data['major_group_engineer'] = salary_data[['major_電機電子工程相關', 'major_工程學科類', 'major_機械工程相關']].sum(axis=1).gt(0).astype(int)
salary_data['major_group_math_and_stats'] =  salary_data[['major_數學及電算機科學學科類', 'major_數理統計相關', 'major_其他數學及電算機科學相關', 'major_應用數學相關', 'major_統計學相關']].sum(axis=1).gt(0).astype(int)
salary_data['major_group_commercial'] = salary_data[['major_商業及管理學科類', 'major_一般商業學類']].sum(axis=1).gt(0).astype(int)

major_group = ['major_group_infomation', 'major_group_engineer', 'major_group_math_and_stats', 'major_group_commercial']
test_df = t_test_binary_col(salary_data, major_group, 'min_salary_K', 0.1)
test_df 

,name,mean min_salary_K_with,mean min_salary_K_not_with,mean_diff,p_value
1,major_group_engineer,55.94183,48.689273,7.252557,0.073213


## 4. 年資 v.s. 薪資

In [19]:
periods = list(data['periodDesc'].unique())
test_df = t_test(salary_data, 'periodDesc', periods, 'min_salary_K', 0.1)
test_df 

,periodDesc,mean min_salary_K_with,mean min_salary_K_not_with,mean_diff,p_value
0,經歷不拘,43.417291,55.559060,-12.141769,2.801092e-07
2,1年以上,41.304053,51.397622,-10.093570,7.485589e-07
4,5年以上,89.969841,47.173994,42.795847,1.339979e-04
3,3年以上,61.311007,48.037636,13.273371,1.284618e-03


In [20]:
test_df = t_test(salary_data, 'periodDesc', periods, 'max_salary_K', 0.1)
test_df 

,periodDesc,mean max_salary_K_with,mean max_salary_K_not_with,mean_diff,p_value
0,經歷不拘,67.237425,90.910178,-23.672753,6.390698e-07
4,5年以上,136.227778,75.955851,60.271927,1.059477e-03
3,3年以上,98.229956,76.822218,21.407737,4.392837e-03


## 5. 縣市 v.s. 薪資

In [21]:
salary_data['region'] = salary_data['jobAddrNoDesc'].str[:3]
regions=list(salary_data['region'].str[:3].unique())

anova(salary_data, 'region', 'min_salary_K')

F統計量: 3.17
p-value: 0.0002
  Multiple Comparison of Means - Tukey HSD, FWER=0.05  
group1 group2 meandiff p-adj    lower    upper   reject
-------------------------------------------------------
   台中市    台北市  13.4162 0.0267    0.7406  26.0918   True
   台中市    台南市   0.3984    1.0  -25.7216  26.5184  False
   台中市    嘉義市   3.2984    1.0  -50.4274  57.0243  False
   台中市    宜蘭縣  -7.3016    1.0   -82.413  67.8099  False
   台中市    彰化縣  -7.2349    1.0  -51.5976  37.1277  False
   台中市    新北市  -1.8571    1.0  -19.1253   15.411  False
   台中市    新竹市   -0.647    1.0  -25.7898  24.4957  False
   台中市    新竹縣  15.3567 0.6763   -8.9417  39.6552  False
   台中市     日本  12.6984    1.0   -62.413  87.8099  False
   台中市    桃園市   2.6762    1.0  -18.2366  23.5889  False
   台中市    美國加  41.0317 0.8535  -34.0797 116.1432  False
   台中市    花蓮縣  -2.3016    1.0   -77.413  72.8099  False
   台中市    高雄市  -3.9516    1.0  -22.2627  14.3595  False
   台北市    台南市 -13.0178 0.8632  -37.1118  11.0762  False
   台北市    嘉義市 -10.117

In [22]:
salary_data['region'] = salary_data['jobAddrNoDesc'].str[:3]
regions=list(salary_data['region'].str[:3].unique())

anova(salary_data, 'region', 'max_salary_K')

F統計量: 3.40
p-value: 0.0001
  Multiple Comparison of Means - Tukey HSD, FWER=0.05  
group1 group2 meandiff p-adj    lower    upper   reject
-------------------------------------------------------
   台中市    台北市  21.6549 0.1851   -3.6187  46.9284  False
   台中市    台南市  -4.7786    1.0  -56.8586  47.3015  False
   台中市    嘉義市  -5.5786    1.0 -112.7011  101.544  False
   台中市    宜蘭縣  -6.1786    1.0 -155.9413 143.5842  False
   台中市    彰化縣  -0.7786    1.0  -89.2321   87.675  False
   台中市    新北市  -1.7947    1.0  -36.2252  32.6358  False
   台中市    新竹市   3.5487    1.0  -46.5828  53.6803  False
   台中市    新竹縣  67.1881 0.0003   18.7401 115.6361   True
   台中市     日本  18.8214    1.0 -130.9413 168.5842  False
   台中市    桃園市   1.0733    1.0  -40.6241  42.7707  False
   台中市    美國加  58.8214 0.9885  -90.9413 208.5842  False
   台中市    花蓮縣  -0.1786    1.0 -149.9413 149.5842  False
   台中市    高雄市  -6.9212    1.0  -43.4312  29.5889  False
   台北市    台南市 -26.4334 0.8468  -74.4738   21.607  False
   台北市    嘉義市 -27.233

In [23]:
region_map = {
    # 北部
    '台北市': '北部',
    '新北市': '北部',
    '基隆市': '北部',
    '桃園市': '北部',
    '新竹市': '北部',
    '新竹縣': '北部',

    # 中部
    '台中市': '中部',
    '苗栗縣': '中部',
    '彰化縣': '中部',
    '南投縣': '中部',
    '雲林縣': '中部',

    # 南部
    '台南市': '南部',
    '高雄市': '南部',
    '嘉義市': '南部',
    '嘉義縣': '南部',
    '屏東縣': '南部',

    # 東部
    '宜蘭縣': '東部',
    '花蓮縣': '東部',
    '台東縣': '東部',

    # 離島 & 外國
    '澎湖縣': '外島',
    '金門縣': '外島',
    '連江縣': '外島'
}

In [24]:
salary_data['region_group'] = salary_data['region'].apply(lambda x: region_map[x] if x in region_map.keys() else '其他')
region_group=list(salary_data['region_group'].str[:3].unique())

anova(salary_data, 'region_group', 'min_salary_K')

F統計量: 4.88
p-value: 0.0008
 Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower    upper  reject
-----------------------------------------------------
    中部     其他  27.3474 0.4395 -16.9461 71.6409  False
    中部     北部  10.7232 0.0261   0.8298 20.6166   True
    中部     南部  -1.9821 0.9943 -15.3916 11.4274  False
    中部     東部  -4.3193 0.9989 -48.6128 39.9742  False
    其他     北部 -16.6242 0.8328 -60.1308 26.8824  False
    其他     南部 -29.3295 0.3695 -73.7677 15.1088  False
    其他     東部 -31.6667 0.6172 -92.9599 29.6265  False
    北部     南部 -12.7053  0.009 -23.2278 -2.1828   True
    北部     東部 -15.0425 0.8777 -58.5491 28.4641  False
    南部     東部  -2.3372 0.9999 -46.7754 42.1011  False
-----------------------------------------------------


In [25]:
anova(salary_data, 'region_group', 'max_salary_K')

F統計量: 3.98
p-value: 0.0036
  Multiple Comparison of Means - Tukey HSD, FWER=0.05  
group1 group2 meandiff p-adj    lower    upper   reject
-------------------------------------------------------
    中部     其他  38.8733 0.7536  -50.2399 127.9866  False
    中部     北部  18.6528 0.0783   -1.2515  38.5572  False
    中部     南部   -6.251 0.9692  -33.2294  20.7273  False
    中部     東部  -3.1267    1.0  -92.2399  85.9866  False
    其他     北部 -20.2205 0.9695 -107.7506  67.3096  False
    其他     南部 -45.1244 0.6383 -134.5288  44.2801  False
    其他     東部    -42.0 0.8835 -165.3146  81.3146  False
    北部     南部 -24.9039 0.0119   -46.074  -3.7338   True
    北部     東部 -21.7795 0.9602 -109.3096  65.7506  False
    南部     東部   3.1244    1.0  -86.2801  92.5288  False
-------------------------------------------------------


## 6. 遠端工作 v.s. 薪資

In [26]:
salary_data['is_remote'] = salary_data['remote_desc'].apply(lambda x: 1 if x=='遠端工作' else 0)
test_df = t_test(salary_data, 'is_remote', [0, 1] , 'min_salary_K', 0.1)
test_df

,is_remote,mean min_salary_K_with,mean min_salary_K_not_with,mean_diff,p_value
0,0,47.249786,63.707547,-16.457761,0.000302
1,1,63.707547,47.249786,16.457761,0.000302


In [27]:
salary_data['is_remote'] = salary_data['remote_desc'].apply(lambda x: 1 if x=='遠端工作' else 0)
test_df = t_test(salary_data, 'is_remote', [0, 1] , 'max_salary_K', 0.1)
test_df

,is_remote,mean max_salary_K_with,mean max_salary_K_not_with,mean_diff,p_value
0,0,74.943026,105.493711,-30.550685,0.000114
1,1,105.493711,74.943026,30.550685,0.000114


## 7. 公司規模 v.s. 薪資

In [60]:
def scale_company(num):

    if num <= 50:
        return '1. 50人以下'
#     if num <= 100:
#         return '2 51~100'
    if num <= 200:
        return '3. 50~200'
#     if num <= 500:
#         return '4. 201~500'
    return '5. >500'

In [61]:
salary_data[['employees']].info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 349 entries, 4 to 1946
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   employees  349 non-null    float64
dtypes: float64(1)
memory usage: 5.5 KB


In [62]:
salary_data['employees'] = salary_data['employees'].fillna(10)
salary_data['company_scale'] = salary_data['employees'].apply(lambda x: scale_company(x))

In [63]:
anova(salary_data, 'company_scale', 'min_salary_K')

F統計量: 13.86
p-value: 0.0000
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
  group1    group2  meandiff p-adj   lower    upper  reject
-----------------------------------------------------------
 1. 50人以下 3. 50~200   -0.616 0.9747  -7.3445  6.1125  False
 1. 50人以下   5. >500 -14.1781    0.0 -20.9066 -7.4496   True
3. 50~200   5. >500 -13.5622 0.0001 -21.0908 -6.0335   True
-----------------------------------------------------------
